# 3. Inference: classify particles and check accuracy

`evaluate` runs a model on particle images and writes `predictions.csv`. With labelled data it also
grades the predictions against the manual classes.

| Section | Input | Model |
|---|---|---|
| A | Erie crops with manual classes (from `preprocess.ipynb`) | released `noise` model |
| B | a raw, **unclassified** FlowCam export | released `noise` model |
| C | Erie crops | your model from `train.ipynb` |
| D | single images, from Python | released model |

The released weights download automatically on first use (115 MB per model) into
`~/.cache/hierroute`.

Requires `pip install hierroute jupyter`. The example data lives at `/data/Hier-Zooplankton` on the lab server; change `DATA_DIR` below if your copy is elsewhere. Everything this notebook writes goes under `WORK_DIR`.

In [ ]:
from pathlib import Path

DATA_DIR = Path("/data/Hier-Zooplankton")        # example dataset (internal)
WORK_DIR = Path.home() / "hierroute_example"      # everything the example notebooks write goes here
WORK_DIR.mkdir(parents=True, exist_ok=True)

CROPS_DIR = WORK_DIR / "erie_crops"                                      # from preprocess.ipynb
RUN_DIR = WORK_DIR / "training_result" / "demo_swin_t_cnn_noise"         # from train.ipynb
RESULTS_DIR = WORK_DIR / "results"

In [ ]:
import pandas as pd
from IPython.display import Image as ShowImage, display
from hierroute import EvaluateOptions, evaluate

## A. How well does the model do on manually classified data?

`labelled=True` means each folder name is the manual class. Folder names are matched to the taxonomy:
- leaf names match directly
- aliases such as `Bosminidae` → `Bosmina`
- catch-alls such as `Detritus` → `Zoop-No`
- coarse IDs such as `Rotifer` → `Rotifer_Unknown`

Folders that don't match (e.g. `MultiClass`, `Unknown`) get predictions but aren't graded.

In [ ]:
erie = evaluate(EvaluateOptions(
    input=CROPS_DIR,
    out_dir=RESULTS_DIR / "erie_noise",
    labelled=True,
    model="noise",            # or "baseline" for clean backgrounds
))

In [ ]:
for name in ["hierarchy_level_metrics.png", "leaf_accuracy.png", "id_outcomes_stacked_bar.png"]:
    display(ShowImage(filename=RESULTS_DIR / "erie_noise" / name, width=750))

`predictions.csv` has one row per image:
- `predicted_class_1`, and the runner-up `predicted_class_2`
- the prediction and the model's confidence at each level of the tree
- the ID outcome (`correct_all` … `incorrect_all`)
- the FlowCam measurements (`csv_*`)

In [ ]:
erie[["image_file", "actual_class", "predicted_class_1", "predicted_class_2",
      "confidence_level_1", "confidence_level_2", "confidence_level_3",
      "predicted_level_1", "predicted_level_2", "predicted_level_3", "correct_all"]].head()

In [ ]:
# Where do the zooplankton folders end up? (rows: manual class, columns: prediction, row %)
zoop = erie[erie["in_taxonomy"] & (erie["predicted_level_1"] == "Zoop-yes")]
pd.crosstab(zoop["actual_class"], zoop["predicted_class_1"], normalize="index").round(2)

In [ ]:
# Low-confidence species calls are good candidates for manual review
erie[erie["confidence_level_3"] < 0.6].sort_values("confidence_level_3")[
    ["image_file", "actual_class", "predicted_class_1", "predicted_class_2", "confidence_level_3"]].head(10)

## B. Classify an unclassified FlowCam export

Here the model reads the raw mosaics and CSVs directly (`input_type="flowcam"`) and crops each particle
in memory. The example is Emily's unclassified export: seven runs from different lakes, where every
CSV row has the placeholder class `Particle`.

The result is one row per particle with its `run`, so counts per sample are a single `crosstab`.

In [ ]:
EMILY = DATA_DIR / "Emily_test_data"

unclassified = evaluate(EvaluateOptions(
    input=EMILY / "Classified_data",            # mosaics
    csv_dir=EMILY / "CSV_Data_unclassified",    # the unclassified export CSVs
    input_type="flowcam",
    out_dir=RESULTS_DIR / "emily_unclassified",
))

In [ ]:
# Particles per predicted class in each run
counts = pd.crosstab(unclassified["predicted_class_1"], unclassified["run"])
counts.loc[counts.sum(axis=1).sort_values(ascending=False).index]

In [ ]:
# Optional: only trust species-level calls above a confidence threshold; otherwise report the group
confident = unclassified["confidence_level_3"].fillna(1) >= 0.9
unclassified["reported_class"] = unclassified["predicted_class_1"].where(
    confident, unclassified["predicted_level_2"].fillna(unclassified["predicted_level_1"]))
unclassified["reported_class"].value_counts()

## C. Use your own trained model

Pass the run folder from `train.ipynb` as `weights`. Its `model_config.json` tells hierroute how to
rebuild the model. The demo model trained on only 300 images per class for 3 epochs, so expect it to
trail the released model.

In [ ]:
if RUN_DIR.exists():
    own = evaluate(EvaluateOptions(input=CROPS_DIR, out_dir=RESULTS_DIR / "erie_demo_model",
                                   labelled=True, weights=RUN_DIR))
    display(ShowImage(filename=RESULTS_DIR / "erie_demo_model" / "hierarchy_level_metrics.png", width=750))
else:
    print(f"No trained model at {RUN_DIR} -- run train.ipynb first.")

In [ ]:
# Released vs demo model: share of graded particles in each ID outcome
outcomes = ["correct_all", "correct_incomplete", "incorrect_l3", "incorrect_l2", "incorrect_all"]
compare = {"released noise model": erie}
if RUN_DIR.exists():
    compare["demo model (train.ipynb)"] = own
pd.DataFrame({name: df.loc[df["in_taxonomy"], outcomes].astype(bool).mean() for name, df in compare.items()}).round(3)

## D. From Python, image by image

`load_model` returns the network and its config. `Predictor` gives the greedy path through the
taxonomy, the confidence at each step and a runner-up. This is useful for building your own tools on
top of hierroute.

In [ ]:
import torch
from PIL import Image
from hierroute import load_model
from hierroute.inference import Predictor, eval_transform

device = "cuda" if torch.cuda.is_available() else "cpu"
model, config = load_model("noise", device=device)
predictor = Predictor(model, device)
transform = eval_transform(config.resolution)

files = [sorted((CROPS_DIR / c).glob("*.tif"))[0] for c in ["Calanoid", "Daphnia", "Rotifer", "Detritus"]]
batch = torch.stack([transform(Image.open(f).convert("L")).repeat(3, 1, 1) for f in files])

for f, pred in zip(files, predictor(batch)):
    path = " > ".join(predictor.id_to_name[n] for n in pred.path[1:])
    confs = ", ".join(f"{c:.3f}" for c in pred.confidences)
    print(f"{f.parent.name:10s} -> {path:40s} confidences {confs} | runner-up {pred.predicted_2}")

## The same from the terminal

```bash
# A: labelled crops
hierroute evaluate ~/hierroute_example/erie_crops --labelled --out results/erie_noise

# B: raw unclassified export
hierroute evaluate /data/Hier-Zooplankton/Emily_test_data/Classified_data --input-type flowcam \
    --csv-dir /data/Hier-Zooplankton/Emily_test_data/CSV_Data_unclassified --out results/emily

# C: your own model
hierroute evaluate ~/hierroute_example/erie_crops --labelled \
    --weights ~/hierroute_example/training_result/demo_swin_t_cnn_noise --out results/demo
```

Optional outputs (per-run bars, confidence thresholds, capped tables, …) are listed under
`hierroute evaluate -h` and in the README.